# Альфа-Банк: ИИ-помощник контакт-центра / RAG

Демо-notebook по мотивам проекта из презентации: помощник оператора ищет релевантные фрагменты базы знаний и формирует подсказку с источниками.

Что внутри:
- synthetic knowledge base по банковским вопросам;
- TF-IDF bi-encoder style retrieval;
- top-K retrieval + confidence;
- простая RAG-ответная функция с citations;
- offline-оценка `Recall@K` на мини golden set.

Это легковесная версия без загрузки большой LLM, чтобы notebook быстро запускался локально.

## Demo scope и production scale

Этот notebook — легковесная демонстрация RAG-помощника, а не production-реализация контакт-центра. Здесь используется маленькая synthetic knowledge base и TF-IDF retrieval, чтобы весь пример запускался быстро и без внешних моделей.

В реальном проекте масштаб был бы другим:
- большая база знаний, шаблоны ответов, документы процессов, история диалогов и контекст клиента;
- BERT/SentenceTransformer embeddings или другой bi-encoder вместо TF-IDF;
- vector database, reranking, deduplication, ACL-фильтры и token budget перед LLM;
- LLM/RAG-генерация с citations, refusal при низкой уверенности и защитой от prompt injection;
- golden set, human evaluation, faithfulness, hallucination rate, latency, cost и мониторинг качества;
- интеграция с рабочим местом оператора и A/B-тесты по AHT, CSAT/VoC и доле полезных подсказок.

Смысл demo: показать основную RAG-цепочку `query → retrieval → context → answer + sources` без приватных банковских данных и тяжелой инфраструктуры.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

knowledge_base = pd.DataFrame([
    {
        "doc_id": "card_block",
        "title": "Блокировка карты",
        "text": "Если клиент потерял карту или подозревает мошенничество, карту нужно немедленно заблокировать в мобильном приложении, интернет-банке или через оператора. После блокировки можно заказать перевыпуск карты.",
    },
    {
        "doc_id": "cashback",
        "title": "Кэшбэк",
        "text": "Кэшбэк начисляется за покупки по карте согласно выбранным категориям. Срок начисления обычно до 10 рабочих дней после подтверждения операции магазином.",
    },
    {
        "doc_id": "mortgage_payment",
        "title": "Досрочное погашение ипотеки",
        "text": "Для досрочного погашения ипотеки клиент оформляет заявку в приложении, выбирает сумму и дату списания. После списания банк пересчитывает график платежей.",
    },
    {
        "doc_id": "loan_delay",
        "title": "Просрочка по кредиту",
        "text": "Если клиент не успевает внести платеж по кредиту, оператор должен рассказать о возможных штрафах, предложить дату ближайшего платежа и проверить доступность реструктуризации.",
    },
    {
        "doc_id": "transfer_limit",
        "title": "Лимиты переводов",
        "text": "Лимиты переводов зависят от типа операции, канала и уровня идентификации клиента. Увеличение лимита может потребовать подтверждения личности или обращения в банк.",
    },
    {
        "doc_id": "travel_insurance",
        "title": "Страховка для путешествий",
        "text": "Страховку для путешествий можно оформить в приложении перед поездкой. Полис покрывает медицинские расходы и может включать дополнительные опции для багажа и отмены поездки.",
    },
])
knowledge_base

## Аналитический вывод

База знаний задает минимальный набор банковских инструкций: блокировка карты, кэшбэк, ипотека, кредит, лимиты и страховка. Для RAG-системы качество базы знаний так же важно, как и качество модели: устаревшие или неполные документы приведут к неправильным подсказкам.

В реальном контакт-центре такие документы нужно версионировать, связывать с правами доступа, регулярно обновлять и проверять на противоречия.

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
doc_matrix = vectorizer.fit_transform(
    (knowledge_base["title"] + ". " + knowledge_base["text"]).tolist()
)


def retrieve(query: str, k: int = 3) -> pd.DataFrame:
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, doc_matrix).ravel()
    top_idx = np.argsort(-scores)[:k]
    result = knowledge_base.iloc[top_idx].copy()
    result["score"] = scores[top_idx]
    result["rank"] = np.arange(1, len(result) + 1)
    return result[["rank", "doc_id", "title", "score", "text"]]

retrieve("клиент потерял карту и боится списаний", k=3)

## Аналитический вывод

TF-IDF retrieval выступает как легковесный bi-encoder baseline: запрос и документы переводятся в векторное пространство, после чего выбираются ближайшие документы по cosine similarity. В примере запрос про потерю карты поднимает документ о блокировке карты, что соответствует ожидаемому поведению помощника.

Для production-версии такой baseline можно заменить на BERT/SentenceTransformer embeddings и добавить cross-encoder reranker, чтобы лучше обрабатывать синонимы и контекст диалога.

In [ ]:
def answer_operator(query: str, k: int = 2, min_confidence: float = 0.08) -> dict:
    hits = retrieve(query, k=k)
    if hits.empty or float(hits["score"].iloc[0]) < min_confidence:
        return {
            "answer": "Недостаточно уверенности по базе знаний. Лучше уточнить вопрос или передать обращение эксперту.",
            "confidence": 0.0,
            "sources": [],
        }

    context = hits.iloc[0]
    sources = hits[["doc_id", "title", "score"]].to_dict("records")
    answer = (
        f"Подсказка оператору: {context['text']} "
        f"Источник: {context['title']} ({context['doc_id']})."
    )
    return {
        "answer": answer,
        "confidence": float(context["score"]),
        "sources": sources,
    }

question = "Клиент спрашивает, когда придет кэшбэк за покупку по карте"
answer_operator(question)

## Аналитический вывод

Функция `answer_operator` реализует упрощенный RAG-паттерн: сначала ищем релевантные документы, затем формируем ответ только на основе найденного контекста и возвращаем источники. Порог `min_confidence` нужен для безопасного отказа, когда база знаний не дает надежного совпадения.

В реальной системе на месте шаблонной сборки ответа может стоять LLM, но принцип остается тем же: в модель передаются исходные тексты/chunks, а не embeddings.

In [ ]:
golden = pd.DataFrame([
    {"query": "потерял карту что делать", "relevant_doc_id": "card_block"},
    {"query": "когда начислят кэшбек", "relevant_doc_id": "cashback"},
    {"query": "как досрочно погасить ипотеку", "relevant_doc_id": "mortgage_payment"},
    {"query": "не успеваю внести платеж по кредиту", "relevant_doc_id": "loan_delay"},
    {"query": "можно увеличить лимит перевода", "relevant_doc_id": "transfer_limit"},
    {"query": "оформить страховку в поездку", "relevant_doc_id": "travel_insurance"},
])


def recall_at_k(k: int = 3) -> float:
    hits = 0
    for row in golden.itertuples(index=False):
        retrieved = set(retrieve(row.query, k=k)["doc_id"])
        hits += int(row.relevant_doc_id in retrieved)
    return hits / len(golden)

for k in [1, 2, 3]:
    print(f"Recall@{k}: {recall_at_k(k):.3f}")

## Аналитический вывод

`Recall@K` показывает, как часто правильный документ попадает в top-K retrieved фрагментов. Для RAG это ключевая метрика: если нужный документ не найден retrieval-слоем, генератор не сможет дать корректный ответ даже при хорошей LLM.

Рост `Recall` при увеличении K ожидаем, но слишком большой K увеличивает шум, latency и стоимость генерации. Поэтому в production обычно добавляют reranking, фильтрацию дублей, ACL-проверки и token budget.

In [ ]:
dialog_context = "Клиент пишет: Я потерял карту, в приложении вижу подозрительную операцию. Что мне делать?"
response = answer_operator(dialog_context, k=3)
print(response["answer"])
print("\nИсточники:")
for source in response["sources"]:
    print(f"- {source['title']} / {source['doc_id']} / score={source['score']:.3f}")

## Аналитический вывод

Финальный пример показывает целевой пользовательский сценарий: оператор получает готовую подсказку и список источников с confidence score. Это снижает время поиска ответа и уменьшает риск галлюцинаций, потому что подсказка привязана к конкретным документам базы знаний.

В production-интерфейсе рядом с ответом стоит показывать источники, confidence и кнопку передачи эксперту при низкой уверенности.